## Qdrand and BGE-M3: main information

### Why Qdrant is our retrieval backend

We chose **Qdrant** because it is the only open‑source vector database that natively fuses dense and sparse vectors into a single hybrid search query, without requiring a separate BM25 index or an external search engine.  
In production benchmarks, hybrid search in Qdrant achieves **91 % recall@10** (compared to 78 % for pure dense and 65 % for pure sparse), and the engine consistently delivers the lowest latency among its peers — a p50 of just **6 ms** on 1 M vectors.  
For our engineering‑standards corpus this means a user can ask *“допустимое отклонение напряжения”* and simultaneously match the exact string *“ПУЭ‑7 п. 1.7.51”* inside a single request, which is exactly the kind of precision that GOST‑heavy retrieval demands.

Equally important, Qdrant runs as a single Rust binary inside Docker with near‑zero operational overhead, and its local‑to‑cloud parity guarantees that the same client code we write today will work unchanged if we later move to a managed deployment.  
Its payload filtering is the strongest in the open‑source category, allowing us to pre‑filter by document ID, section header, or page range before any ANN search happens — a pattern that reduces the search space and improves result quality for technical documents with deep hierarchical metadata.

Crucially for our initial architecture, **Qdrant can generate BM25‑based sparse vectors automatically** from the payload text.  
This means we don’t need the embedding model to output sparse vectors on day one — the database itself provides the lexical precision component of hybrid search, while the model focuses on semantic understanding.  
The database is Apache‑2.0 licensed, hardware‑agnostic, and delivers consistent sub‑10 ms latency on CPU‑only machines, which keeps our ingestion and retrieval pipeline fully vendor‑agnostic and cost‑effective.

Finally, Qdrant has become the consensus recommendation for filter‑heavy RAG workloads in 2026 independent benchmarks: it combines the best developer experience, mature Python client libraries, production‑grade snapshot tooling, and transparent self‑hosted pricing.  
For our pilot, it provides the ideal foundation — a built‑in hybrid search that complements any dense embedding model, a lightweight deployment model that matches our local‑first architecture, and a clear path to cloud when we eventually scale beyond a single machine.

---

### Why BGE‑M3 is our embedding model of choice (for the dense component)

We selected **BGE‑M3** as our first embedding model because it produces exceptionally strong **dense vectors** — the core component for semantic search — while keeping the door open to more advanced retrieval patterns later.  
The dense vector (1024‑dim, L2‑normalized) captures the overall meaning of a chunk reliably, handling paraphrases and synonyms that are abundant in technical regulations (e.g., “автоматический ввод резерва” vs “АВР”).

Although the full PyTorch version of BGE‑M3 can additionally output learned sparse vectors and ColBERT token‑level embeddings, **we start with the official ONNX export** provided by BAAI, which is streamlined to deliver only the dense representation.  
This keeps our initial deployment simple, fully vendor‑agnostic, and highly performant.

We deliberately pair this dense‑only model with **Qdrant’s built‑in BM25 sparse vector generation**.  
The database automatically extracts lexical signals from the same chunk text and fuses them with the dense vectors in hybrid search.  
This gives us the best of both worlds from day one:
- The dense vector (from BGE‑M3) handles semantic paraphrasing,
- The BM25 sparse vector (from Qdrant) locks onto exact references like “ПУЭ‑7, п. 1.7.51” and domain‑specific abbreviations.

In the future, if our retrieval quality benchmarks reveal a need for even stronger lexical matching, we can either upgrade to the full PyTorch BGE‑M3 (which outputs a learned sparse vocabulary) or adopt a dedicated re‑ranker stage with ColBERT — all without changing the database layer.  
For the current milestone, the official ONNX BGE‑M3 + Qdrant hybrid search provides the ideal balance between proven reliability, resource efficiency, and a clear upgrade path.



In [1]:
# Enable autoreload to automatically pick up changes in local modules
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import json
from pathlib import Path
from IPython.display import display, Markdown

from doc_agent.configs.settings import settings
from doc_agent.utils.logger import setup_logger

# Configure logging
logger = setup_logger(
    name="009_retrival_stack_rationale", 
    level=logging.INFO,
    log_file="retrival_test.log"
)
logger.info("Environment initialized.")

2026-05-20 14:36:55 |     INFO | 009_retrival_stack_rationale:103218203.py:15 - Environment initialized.


## Download the bge-m3 model 

In [3]:
from pathlib import Path
from huggingface_hub import snapshot_download


model_dir = Path("../models/bge_m3_onnx")

In [ ]:
# Download the official ONNX export from BAAI/bge-m3
snapshot_download(
    repo_id="BAAI/bge-m3",
    allow_patterns=["onnx/*"],        
    local_dir=str(model_dir),
)

logger.info(f"ONNX model downloaded to: {model_dir}")

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

onnx/model.onnx_data:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

onnx/model.onnx:   0%|          | 0.00/725k [00:00<?, ?B/s]

onnx/sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

onnx/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

Constant_7_attr__value:   0%|          | 0.00/65.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

2026-05-19 22:03:55 |     INFO | 009_retrival_stack_rationale:3516417873.py:13 - ONNX model downloaded to: ../models/bge_m3_onnx


## Load tokenizer and create ONNX session.

In [ ]:
from transformers import AutoTokenizer
import onnxruntime as ort

# Load tokenizer from the downloaded ONNX folder inside model_dir
tokenizer = AutoTokenizer.from_pretrained(str(model_dir / "onnx"))
logger.info("Tokenizer loaded – vocab size: %d", tokenizer.vocab_size)

# Create ONNX Runtime session (CPU / CoreML on macOS)
providers = ['CPUExecutionProvider']
if 'CoreMLExecutionProvider' in ort.get_available_providers():
    providers = ['CoreMLExecutionProvider'] + providers

session = ort.InferenceSession(
    str(model_dir / "onnx" / "model.onnx"),
    providers=providers
)
logger.info("ONNX session created – providers: %s", session.get_providers())

2026-05-20 14:44:10 |     INFO | 009_retrival_stack_rationale:1920336892.py:6 - Tokenizer loaded – vocab size: 250002
2026-05-20 14:44:45 |     INFO | 009_retrival_stack_rationale:1920336892.py:17 - ONNX session created – providers: ['CPUExecutionProvider']


## Simple test of tokenizer and ONNX model

Give a sample and tokenize with the same tokenizer:

In [9]:
sample = [
    "Сечение заземляющего проводника должно быть не менее 6 мм² для меди.",
    "Требования к автоматическому вводу резерва."
]

In [10]:
inputs = tokenizer(
        sample,
        padding=True,
        truncation=True,
        max_length=8192,
        return_tensors="np"
    )
logger.info(f"Inputs: {inputs}")

2026-05-20 16:34:32 |     INFO | 009_retrival_stack_rationale:415405393.py:8 - Inputs: {'input_ids': array([[     0,   6891,  66639,     61,  20638,   2033,  43155,  43903,
          5119,  23494,   3505,     77,  22947,    305,  33930,    304,
           518,  67142,      5,      2],
       [     0, 124764,  68146,    718,  22251, 115987,    105,     49,
          8778,    105,  27833,     59,      5,      2,      1,      1,
             1,      1,      1,      1]]), 'attention_mask': array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]])}


In [ ]:
# `inputs` already contains 'input_ids' and 'attention_mask'
# The ONNX model expects exactly these keys, so we pass it directly
outputs = session.run(None, inputs)

In [21]:
outputs

[array([[[-1.4026697 ,  0.5533192 , -0.63375646, ..., -0.08195213,
           0.6152495 ,  0.8020947 ],
         [-1.0725266 ,  0.471295  , -0.31263122, ..., -0.29618394,
          -0.11088699,  0.50960773],
         [-0.40087098,  0.445271  , -0.07490529, ..., -0.19965222,
          -0.01669711,  0.82529294],
         ...,
         [-0.17903718,  0.66544145, -0.5739061 , ..., -0.10737008,
          -0.44282943,  0.22567186],
         [-0.16577865,  0.5511606 ,  0.36064354, ...,  0.27543092,
          -0.86752146,  0.1171554 ],
         [ 0.45977628,  1.4006431 ,  0.6359568 , ...,  0.67222106,
          -0.60728246, -0.2569781 ]],
 
        [[-0.683589  ,  0.21755745, -0.92152786, ...,  0.3879603 ,
           0.28482106, -0.23890658],
         [-0.33351362, -0.5541646 , -0.7705909 , ...,  0.21589911,
          -0.37937704, -0.27949166],
         [-0.8385067 , -0.39887324, -0.6391684 , ...,  0.4096346 ,
          -0.0659413 , -0.21423577],
         ...,
         [-0.27259767,  0.6887709

In [25]:
print("Number of output tensors:", len(outputs[0]))
print("last_hidden shape:", outputs[0].shape)   # (2, 20, 1024)
print("dtype:", outputs[0].dtype)
print("First 5 values of [CLS] token for first sentence:")
print(outputs[0][0, 0, :5])

Number of output tensors: 2
last_hidden shape: (2, 20, 1024)
dtype: float32
First 5 values of [CLS] token for first sentence:
[-1.4026697   0.5533192  -0.63375646 -0.9983401   0.25003785]


In [26]:
# The [CLS] token is the very first token (index 0)
cls_raw = outputs[0][:, 0, :]        # shape (2, 1024)

print("CLS raw shape:", cls_raw.shape)
print("First 5 values of sentence 1:", cls_raw[0, :5])
print("First 5 values of sentence 2:", cls_raw[1, :5])
print("Raw L2 norms:", np.linalg.norm(cls_raw, axis=1))

CLS raw shape: (2, 1024)
First 5 values of sentence 1: [-1.4026697   0.5533192  -0.63375646 -0.9983401   0.25003785]
First 5 values of sentence 2: [-0.683589    0.21755745 -0.92152786  0.45429853 -0.7643662 ]
Raw L2 norms: [26.605343 26.117596]


In [27]:
import numpy as np

# Normalize to unit length
cls_norm = cls_raw / np.linalg.norm(cls_raw, axis=1, keepdims=True)

print("Normalized CLS shape:", cls_norm.shape)
print("L2 norms after normalization:", np.linalg.norm(cls_norm, axis=1))
# Should print [1. 1.]

print("\nFirst 5 values of first normalized vector:")
print(cls_norm[0, :5])

Normalized CLS shape: (2, 1024)
L2 norms after normalization: [0.99999994 0.99999994]

First 5 values of first normalized vector:
[-0.05272135  0.0207973  -0.02382065 -0.03752404  0.00939803]


In [28]:
# With normalized vectors, cosine = dot product
sim = np.dot(cls_norm[0], cls_norm[1])
print(f"Cosine similarity between the two sentences: {sim:.4f}")

Cosine similarity between the two sentences: 0.4018


## Install qdrant-client and check connection

In [ ]:
from qdrant_client import QdrantClient

# Connect to the local Qdrant (running in Docker)
client = QdrantClient(host="localhost", port=6333)

# Simple health check – if Qdrant is down, this will raise a connection error
collections = client.get_collections()
logger.info(f"Qdrant reachable – collections: {collections.collections}" )

2026-05-20 18:06:06 |     INFO | 009_retrival_stack_rationale:1818284275.py:7 - Qdrant reachable – collections: []


## Take payloads

In [47]:
payload_path = Path.cwd().parents[0] / "data" / "02_interim" / "pue_1_3" / "06_payloads" / "pue_1_3_payloads.json"
logger.info(f"Payload path: {payload_path}")

2026-05-20 18:32:47 |     INFO | 009_retrival_stack_rationale:1656409860.py:2 - Payload path: /Volumes/SSD/AI/doc_agent/data/02_interim/pue_1_3/06_payloads/pue_1_3_payloads.json


In [48]:
payloads_json = json.loads(payload_path.read_text(encoding="utf-8"))
logger.info(f"Payload json: {payloads_json}")

2026-05-20 18:32:55 |     INFO | 009_retrival_stack_rationale:4089742323.py:2 - Payload json: [{'chunk_id': '689b20648b228b1c', 'text': '16', 'metadata': {'doc_id': 'pue_1_3', 'source_pdf': '/Volumes/SSD/AI/doc_agent/data/01_raw/pue_1_3.pdf', 'pages': ['page_0001'], 'page_artifacts': {'page_0001': '02_renders_png/page_0001_highres.png'}, 'h1': None, 'h2': None, 'h3': None, 'h4': None}}, {'chunk_id': '34a801ac0fcb0db7', 'text': '# Раздел 1. ОБЩИЕ ПРАВИЛА', 'metadata': {'doc_id': 'pue_1_3', 'source_pdf': '/Volumes/SSD/AI/doc_agent/data/01_raw/pue_1_3.pdf', 'pages': ['page_0001'], 'page_artifacts': {'page_0001': '02_renders_png/page_0001_highres.png'}, 'h1': 'Раздел 1. ОБЩИЕ ПРАВИЛА', 'h2': None, 'h3': None, 'h4': None}}, {'chunk_id': '645338ca90a34f5a', 'text': '## Глава 1.1 ОБЩАЯ ЧАСТЬ\n\nУтверждено Министерством энергетики Российской Федерации Приказ от 8 июля 2002 г. № 204. Введено в действие с 1 января 2003 г.', 'metadata': {'doc_id': 'pue_1_3', 'source_pdf': '/Volumes/SSD/AI/doc_age

## Generate dense embeddings for the first 3 payload chunks

In [50]:
# Take the first 3 payloads from the loaded JSON
sample_payloads = payloads_json[:3]
sample_texts = [p["text"] for p in sample_payloads]

# Tokenize (reusing the tokenizer we already loaded)
inputs = tokenizer(sample_texts, padding=True, truncation=True, max_length=8192, return_tensors="np")

# Run ONNX model
outputs = session.run(None, inputs)
last_hidden = outputs[0]          # shape: (3, seq_len, 1024)

# Extract [CLS] token (first token) – this is the dense embedding before normalization
cls_raw = last_hidden[:, 0, :]    # shape: (3, 1024)

# L2‑normalize (unit length), so later dot product = cosine similarity
dense_vecs = cls_raw / np.linalg.norm(cls_raw, axis=1, keepdims=True)

# Log the outcome
logger.info(f"Dense embeddings shape: {dense_vecs.shape}")
logger.info(f"L2 norms (should all be 1.0): {np.linalg.norm(dense_vecs, axis=1)}")

2026-05-20 19:21:10 |     INFO | 009_retrival_stack_rationale:341449338.py:19 - Dense embeddings shape: (3, 1024)
2026-05-20 19:21:10 |     INFO | 009_retrival_stack_rationale:341449338.py:20 - L2 norms (should all be 1.0): [1. 1. 1.]


## Create Qdrant collection and upsert points

In [51]:
from qdrant_client.models import Distance, VectorParams

# Name for this test collection
collection_name = "pue_dense_test"

# Remove it if it already exists (so the notebook is re‑runnable)
try:
    client.delete_collection(collection_name)
    logger.info(f"Deleted existing collection '{collection_name}'")
except Exception:
    pass

# Create a fresh collection for 1024‑dimensional dense vectors
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=1024, distance=Distance.DOT),
)

logger.info(f"Collection '{collection_name}' created (size 1024, distance DotProduct)")

2026-05-20 19:28:02 |     INFO | 009_retrival_stack_rationale:3133322956.py:9 - Deleted existing collection 'pue_dense_test'
2026-05-20 19:28:04 |     INFO | 009_retrival_stack_rationale:3133322956.py:19 - Collection 'pue_dense_test' created (size 1024, distance DotProduct)


In [52]:
from qdrant_client.models import PointStruct

# Each payload chunk becomes one Qdrant point
points = []
for i, payload in enumerate(sample_payloads):
    points.append(
        PointStruct(
            id=i,
            vector=dense_vecs[i].tolist(),   
            payload={
                "text": payload["text"],
                **payload["metadata"]          
            }
        )
    )

client.upsert(
    collection_name=collection_name,
    points=points,
    wait=True,                                
)

logger.info(f"Uploaded {len(points)} points (chunks) to collection '{collection_name}'")

2026-05-20 19:34:34 |     INFO | 009_retrival_stack_rationale:4106981403.py:23 - Uploaded 3 points (chunks) to collection 'pue_dense_test'


## Test search

In [57]:
from qdrant_client.models import QueryRequest, SearchRequest

# Build a query that should match one of the PUE chunks
query_text = "допустимое сечение заземляющего проводника"

# Tokenize and embed the query
query_inputs = tokenizer([query_text], padding=True, truncation=True, max_length=8192, return_tensors="np")
query_outputs = session.run(None, query_inputs)
query_vec = query_outputs[0][:, 0, :]                               # CLS token
query_vec = query_vec / np.linalg.norm(query_vec, axis=1, keepdims=True)

# New Qdrant query API (available in client version 1.9+)
hits = client.query_points(
    collection_name=collection_name,
    query=query_vec[0].tolist(),
    limit=3,
    with_payload=True,
)

# Log results
logger.info(f"Query: {query_text}")
for hit in hits.points:            
    logger.info(f"Score: {hit.score:.4f} | h1: {hit.payload.get('h1')} | h2: {hit.payload.get('h2')} | text: {hit.payload['text'][:80]}")

2026-05-20 19:46:40 |     INFO | 009_retrival_stack_rationale:112257112.py:21 - Query: допустимое сечение заземляющего проводника
2026-05-20 19:46:40 |     INFO | 009_retrival_stack_rationale:112257112.py:23 - Score: 0.3835 | h1: Раздел 1. ОБЩИЕ ПРАВИЛА | h2: Глава 1.1 ОБЩАЯ ЧАСТЬ | text: ## Глава 1.1 ОБЩАЯ ЧАСТЬ

Утверждено Министерством энергетики Российской Федерац
2026-05-20 19:46:40 |     INFO | 009_retrival_stack_rationale:112257112.py:23 - Score: 0.3424 | h1: Раздел 1. ОБЩИЕ ПРАВИЛА | h2: None | text: # Раздел 1. ОБЩИЕ ПРАВИЛА
2026-05-20 19:46:40 |     INFO | 009_retrival_stack_rationale:112257112.py:23 - Score: 0.2722 | h1: None | h2: None | text: 16


## Create a collection with automatic BM25 sparse vectors

In [58]:
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, Modifier

# New collection for hybrid search
hybrid_collection_name = "pue_hybrid_test"

# Remove if it already exists (for re-runnable notebook cells)
try:
    client.delete_collection(hybrid_collection_name)
    logger.info(f"Deleted existing collection '{hybrid_collection_name}'")
except Exception:
    pass

# Create collection with dense vectors and automatic BM25 sparse vector generation
client.create_collection(
    collection_name=hybrid_collection_name,
    vectors_config=VectorParams(
        size=1024,               # BGE-M3 dense vector dimension
        distance=Distance.DOT    # DotProduct – our vectors are L2-normalized
    ),
    sparse_vectors_config={
        "bm25": SparseVectorParams(
            modifier=Modifier.IDF,   # Classic IDF weighting for BM25
        )
    },
)

logger.info(f"Collection '{hybrid_collection_name}' created with dense (1024, Dot) and auto-BM25 sparse vectors")

2026-05-21 16:21:26 |     INFO | 009_retrival_stack_rationale:1522730491.py:9 - Deleted existing collection 'pue_hybrid_test'
2026-05-21 16:21:28 |     INFO | 009_retrival_stack_rationale:1522730491.py:27 - Collection 'pue_hybrid_test' created with dense (1024, Dot) and auto-BM25 sparse vectors


## Upload the three test points (dense + payload with text)

In [59]:
from qdrant_client.models import PointStruct

points = []
for i, payload in enumerate(sample_payloads):
    points.append(
        PointStruct(
            id=i,
            vector=dense_vecs[i].tolist(),        # same dense vector as before
            payload={
                "text": payload["text"],
                **payload["metadata"]
            }
        )
    )

client.upsert(
    collection_name=hybrid_collection_name,
    points=points,
    wait=True,
)

logger.info(f"Uploaded {len(points)} points to '{hybrid_collection_name}' – sparse vectors generated automatically")

2026-05-21 16:22:33 |     INFO | 009_retrival_stack_rationale:3751752495.py:22 - Uploaded 3 points to 'pue_hybrid_test' – sparse vectors generated automatically


## Hybrid search (dense + BM25)

In [62]:
from collections import Counter
from qdrant_client import models

# The query text
query_text = "допустимое сечение заземляющего проводника"

# --- Dense vector (as before) ---
query_inputs = tokenizer([query_text], padding=True, truncation=True, max_length=8192, return_tensors="np")
query_outputs = session.run(None, query_inputs)
dense_vec = query_outputs[0][:, 0, :]
dense_vec = dense_vec / np.linalg.norm(dense_vec, axis=1, keepdims=True)
dense_list = dense_vec[0].tolist()

# --- Sparse vector from query text ---
# Tokenize and count token frequencies to simulate a BM25 sparse vector
query_tokens = tokenizer.encode(query_text)
token_counts = Counter(query_tokens)
sparse_indices = list(token_counts.keys())
sparse_values = [float(token_counts[i]) for i in sparse_indices]

# Create a sparse vector object
sparse_vector = models.SparseVector(indices=sparse_indices, values=sparse_values)

# --- Hybrid search with prefetch and RRF fusion ---
hits = client.query_points(
    collection_name=hybrid_collection_name,
    prefetch=[
        # Prefetch 1: Dense search
        models.Prefetch(
            query=dense_list,
            limit=10,
        ),
        # Prefetch 2: Sparse search using the named "bm25" vector
        models.Prefetch(
            query=sparse_vector,
            using="bm25",
            limit=10,
        ),
    ],
    # Merge dense and sparse results with Reciprocal Rank Fusion
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=3,
    with_payload=True,
)

# Log results
logger.info(f"Hybrid query: {query_text}")
for hit in hits.points:
    logger.info(f"Score: {hit.score:.4f} | h1: {hit.payload.get('h1')} | h2: {hit.payload.get('h2')} | text: {hit.payload['text'][:80]}")

2026-05-21 16:36:48 |     INFO | 009_retrival_stack_rationale:664758273.py:47 - Hybrid query: допустимое сечение заземляющего проводника
2026-05-21 16:36:48 |     INFO | 009_retrival_stack_rationale:664758273.py:49 - Score: 0.5000 | h1: Раздел 1. ОБЩИЕ ПРАВИЛА | h2: Глава 1.1 ОБЩАЯ ЧАСТЬ | text: ## Глава 1.1 ОБЩАЯ ЧАСТЬ

Утверждено Министерством энергетики Российской Федерац
2026-05-21 16:36:48 |     INFO | 009_retrival_stack_rationale:664758273.py:49 - Score: 0.3333 | h1: Раздел 1. ОБЩИЕ ПРАВИЛА | h2: None | text: # Раздел 1. ОБЩИЕ ПРАВИЛА
2026-05-21 16:36:48 |     INFO | 009_retrival_stack_rationale:664758273.py:49 - Score: 0.2500 | h1: None | h2: None | text: 16


## Simple Retrieval Testing of library
Validate hybrid search after indexing.

In [2]:
import logging
from doc_agent.utils.logger import setup_logger
from doc_agent.configs.settings import settings
from doc_agent.indexing.embedder import Embedder
from doc_agent.retrieval.hybrid_searcher import HybridSearcher

# Configure the root logger once (call exactly once in the whole process)
setup_logger(name="", level=logging.INFO)

# Get a module‑specific logger (optional but clean)
logger = logging.getLogger(__name__)

logger.info("Loading embedding model and connecting to Qdrant...")
embedder = Embedder()
searcher = HybridSearcher(embedder)
logger.info("Ready.\n")

queries = [
    "заземляющие устройства",
    "сечение проводов",
    "допустимые токи",
    "классификация помещений по опасности",
]

for q in queries:
    logger.info("Query: %s", q)
    results = searcher.search(
        q,
        collection_name=settings.QDRANT_COLLECTION_NAME,
        limit=5,
    )
    for i, res in enumerate(results, 1):
        score = res["score"]
        text = res["text"][:300].replace("\n", " ")
        doc = res["metadata"].get("doc_id", "?")
        pages = res["metadata"].get("pages", [])
        logger.info(
            "%d. score=%.4f | doc=%s | pages=%s",
            i, score, doc, pages,
        )
        logger.info("   %s...", text)
    logger.info("-" * 80)

2026-05-26 23:05:05 |     INFO | __main__:3945113467.py:13 - Loading embedding model and connecting to Qdrant...
2026-05-26 23:05:05 |     INFO | doc_agent.indexing.embedder:embedder.py:40 - Loading BGE-M3 tokenizer from: /Volumes/SSD/AI/doc_agent/models/bge_m3_onnx/onnx


2026-05-26 23:05:06 |     INFO | doc_agent.indexing.embedder:embedder.py:44 - Tokenizer loaded – vocab size: 250002
2026-05-26 23:05:41 |     INFO | doc_agent.indexing.embedder:embedder.py:56 - ONNX session created – providers: ['CPUExecutionProvider']
2026-05-26 23:05:41 |     INFO | doc_agent.retrieval.hybrid_searcher:hybrid_searcher.py:51 - HybridSearcher connected to Qdrant at http://localhost:6333
2026-05-26 23:05:41 |     INFO | __main__:3945113467.py:16 - Ready.

2026-05-26 23:05:41 |     INFO | __main__:3945113467.py:26 - Query: заземляющие устройства
2026-05-26 23:05:41 |     INFO | httpx:_client.py:1025 - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-05-26 23:05:41 |     INFO | doc_agent.indexing.embedder:embedder.py:100 - Successfully embedded 1 texts – output shape: (1, 1024)
2026-05-26 23:05:41 |     INFO | httpx:_client.py:1025 - HTTP Request: POST http://localhost:6333/collections/pue_chunks/points/query "HTTP/1.1 200 OK"
2026-05-26 23:05:41 |     INFO |

In [3]:
import pandas as pd
import logging
from doc_agent.utils.logger import setup_logger
from doc_agent.configs.settings import settings
from doc_agent.indexing.embedder import Embedder
from doc_agent.retrieval.hybrid_searcher import HybridSearcher

# Keep logging only for important messages (model loading, errors)
setup_logger(name="", level=logging.WARNING)

print("Loading model and connecting to Qdrant...")
embedder = Embedder()
searcher = HybridSearcher(embedder)
print("Ready.\n")

queries = [
    "заземляющие устройства",
    "сечение проводов",
    "допустимые токи",
    "классификация помещений по опасности",
]

rows = []
for q in queries:
    results = searcher.search(
        q,
        collection_name=settings.QDRANT_COLLECTION_NAME,
        limit=5,          # top‑5 after fusion
        dense_limit=10,
        sparse_limit=10,
    )
    for rank, res in enumerate(results, 1):
        meta = res["metadata"]
        rows.append({
            "query": q,
            "rank": rank,
            "score": round(res["score"], 4),
            "doc_id": meta.get("doc_id", ""),
            "pages": ", ".join(meta.get("pages", [])),
            "h1": meta.get("h1", ""),
            "h2": meta.get("h2", ""),
            "h3": meta.get("h3", ""),
            "h4": meta.get("h4", ""),
            "text_snippet": res["text"][:200].replace("\n", " "),
        })

df = pd.DataFrame(rows)
df.head(20)  # show first 20 rows in a notebook

Loading model and connecting to Qdrant...
Ready.



,query,rank,score,doc_id,pages,h1,h2,h3,h4,text_snippet
0,заземляющие устройства,1,0.5000,pue_1.1-1.3,page_0008,Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.3 ЭКОНОМИЧЕСКОЙ,Область применения,1.3.7.,#### 1.3.7. Требования к нормальным нагрузкам...
1,заземляющие устройства,2,0.3333,pue_1.1-1.3,page_0027,Допустимый длительный ток для шин прямоугольно...,Глава 1.4 ВЫБОР ЭЛЕКТРИЧЕСКИХ АППАРАТОВ И ПРОВ...,NaN,NaN,## Глава 1.4 ВЫБОР ЭЛЕКТРИЧЕСКИХ АППАРАТОВ И П...
2,заземляющие устройства,3,0.2500,pue_1.1-1.3,"page_0005, page_0006",Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.2 ЭЛЕКТРОСНАБЖЕНИЕ И ЭЛЕКТРИЧЕСКИЕ СЕТИ,Общие требования,NaN,### Общие требования #### 1.2.11. При проекти...
3,заземляющие устройства,4,0.2000,pue_1.1-1.3,page_0027,Допустимый длительный ток для шин прямоугольно...,Глава 1.4 ВЫБОР ЭЛЕКТРИЧЕСКИХ АППАРАТОВ И ПРОВ...,Общие требования,NaN,### Общие требования #### 1.4.2. По режиму КЗ...
4,заземляющие устройства,5,0.1667,pue_1.1-1.3,page_0004,Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.1 ОБЩАЯ ЧАСТЬ,Общие указания по устройству электроустановок,1.1.34.,"#### 1.1.34. В жилых, общественных и других п..."
5,сечение проводов,1,0.5000,pue_1.1-1.3,page_0025,Допустимый длительный ток для шин прямоугольно...,NaN,Выбор сечения проводников по экономической пло...,NaN,### Выбор сечения проводников по экономической...
6,сечение проводов,2,0.3333,pue_23_25,page_0003,Допустимый длительный ток для шин прямоугольно...,NaN,Выбор сечения проводников по экономической пло...,NaN,### Выбор сечения проводников по экономической...
7,сечение проводов,3,0.2500,pue_1.1-1.3,page_0025,Допустимый длительный ток для шин прямоугольно...,NaN,Выбор сечения проводников по экономической пло...,1.3.25.,#### 1.3.25. Сечения проводников должны быть ...
8,сечение проводов,4,0.2000,pue_1.1-1.3,"page_0007, page_0008",Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.3 ЭКОНОМИЧЕСКОЙ,Область применения,NaN,### Область применения #### 1.3.1. Настоящая ...
9,сечение проводов,5,0.1667,pue_1.1-1.3,"page_0021, page_0022, page_0023",Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.3 ЭКОНОМИЧЕСКОЙ,Поправочный коэффициент a на сечение кабеля,NaN,### Поправочный коэффициент a на сечение кабел...


## Simple test of AnswerGeneratorAgent

In [4]:
from doc_agent.agents.answer_generator import AnswerGeneratorAgent

# Load the answer‑generation system prompt
prompt_path = settings.PROMPTS_DIR / "answer_generation.md"
rag_prompt = prompt_path.read_text(encoding="utf-8")

# Create the agent (reuses your existing searcher)
answer_agent = AnswerGeneratorAgent(searcher)

# Test queries
test_questions = [
    "Какое минимальное сечение заземляющего проводника?",
    "классификация помещений по опасности",
]

for q in test_questions:
    result = answer_agent.generate(q, rag_prompt)
    print(f"Q: {q}")
    print(f"A: {result.answer}")
    for src in result.sources:
        print(f"  [{src.index}] {src.doc_id}, {src.pages}, {src.section}")
    print("-" * 80)

Q: Какое минимальное сечение заземляющего проводника?
A: В предоставленных документах ответ не найден
  [1] pue_1.1-1.3, page_0007,page_0008, Раздел 1. ОБЩИЕ ПРАВИЛА / Глава 1.3 ЭКОНОМИЧЕСКОЙ / Область применения
  [2] pue_1.1-1.3, page_0026,page_0027, Допустимый длительный ток для шин прямоугольного сечения / Глава 1.3. Выбор проводников по нагреву, эконом. плотности тока и по условиям короны
  [3] pue_1.1-1.3, page_0015, Раздел 1. ОБЩИЕ ПРАВИЛА / Глава 1.3 ЭКОНОМИЧЕСКОЙ / Допустимый длительный ток для кабелей с алюминиевыми жилами с бумажной пропитанной маслоканифольной и нестекающей массами изоляцией в свинцовой или алюминиевой оболочке, прокладываемых в земле
  [4] pue_1.1-1.3, page_0010,page_0011,page_0012,page_0013,page_0014, Раздел 1. ОБЩИЕ ПРАВИЛА / Глава 1.3 ЭКОНОМИЧЕСКОЙ / Допустимые длительные токи для проводов, шнуров и кабелей с резиновой или пластмассовой изоляцией
  [5] pue_1.1-1.3, page_0015,page_0016,page_0017,page_0018,page_0019, Раздел 1. ОБЩИЕ ПРАВИЛА / Глава 1.3 ЭК

## Simple test of Agentic RAG 

In [7]:
from doc_agent.agents.answer_generator import AnswerGeneratorAgent
from doc_agent.agents.agentic_answer_agent import AgenticAnswerAgent

# Create the agents
answer_gen = AnswerGeneratorAgent(searcher)
agentic = AgenticAnswerAgent(searcher, answer_gen)

# Load the prompt
prompt = (settings.PROMPTS_DIR / "answer_generation.md").read_text(encoding="utf-8")

# A complex question
question = "Какие требования к выбору проводников по экономической плотности тока?"

result = agentic.generate(question, prompt)
print("ANSWER:", result.answer)
print("\nSOURCES:")
for s in result.sources:
    print(f"  [{s.index}] {s.doc_id}, {s.pages}, {s.section}")

2026-05-26 23:41:34 |  WARNING | doc_agent.agents.agentic_answer_agent:agentic_answer_agent.py:146 - All retrieval attempts exhausted for 'Какие нормативные значения экономической плотности тока для '.
2026-05-26 23:42:43 |  WARNING | doc_agent.agents.agentic_answer_agent:agentic_answer_agent.py:146 - All retrieval attempts exhausted for 'Как влияют группировка проводников, их взаимное расположение'.
2026-05-26 23:45:04 |  WARNING | doc_agent.reasoning.faithfulness_checker:faithfulness_checker.py:77 - 10 unsupported claim(s) found.
2026-05-26 23:45:04 |  WARNING | doc_agent.agents.agentic_answer_agent:agentic_answer_agent.py:94 - Answer contains 10 unsupported claim(s); removing them.
ANSWER: Сечения проводников должны быть проверены по экономической плотности тока. Экономически целесообразное сечение $S$, мм$^2$ определяется из соотношения

$$S=\dfrac{I}{J_{эк}}$$

где $I$ — расчетный ток в час максимума энергосистемы, А; $J_{эк}$ — нормированное значение экономической плотности тока,